# 01 Data Preprocessing

This notebook prepares the image sentiment dataset for model training. It includes dataset loading, label remapping for binary classification, train/validation/test split, data augmentation, ImageNet normalization, DataLoader creation, and class distribution checks.

In [ ]:
from google.colab import drive
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
base_dir   = "/content/drive/MyDrive/MIS 548/Project/sentiment_dataset"
batch_size = 32
num_workers = 0

train_ratio = 0.70
val_ratio   = 0.15
test_ratio  = 0.15

In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize(256),           # resize shortest side to 256
    transforms.CenterCrop(224),       # square crop to 224x224 for AlexNet
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_test_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [ ]:
full_dataset = datasets.ImageFolder(root=base_dir, transform=train_transforms)

# add the remapping immediately after
label_map = {
    0: 0,  # angry       → negative
    1: 1,  # happy       → positive
    2: 0,  # melancholic → negative
    3: 0,  # sad         → negative
}

full_dataset.targets = [label_map[label] for label in full_dataset.targets]
full_dataset.samples = [(path, label_map[label]) for path, label in full_dataset.samples]

total      = len(full_dataset)
train_size = int(total * train_ratio)
val_size   = int(total * val_ratio)
test_size  = total - train_size - val_size  # absorbs rounding remainder

In [ ]:
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset

# Get all labels from the dataset
labels = [label for _, label in full_dataset.samples]

# First split off train
train_idx, temp_idx = train_test_split(
    range(total), test_size=(val_ratio + test_ratio),
    stratify=labels, random_state=42
)

# Split remaining into val and test
temp_labels = [labels[i] for i in temp_idx]
val_idx, test_idx = train_test_split(
    temp_idx, test_size=test_ratio / (val_ratio + test_ratio),
    stratify=temp_labels, random_state=42
)

train_dataset = Subset(full_dataset, train_idx)
val_dataset   = Subset(full_dataset, val_idx)
test_dataset  = Subset(full_dataset, test_idx)

In [ ]:
val_base  = datasets.ImageFolder(root=base_dir, transform=val_test_transforms)
val_base.targets = [label_map[label] for label in val_base.targets]
val_base.samples = [(path, label_map[label]) for path, label in val_base.samples]

test_base = datasets.ImageFolder(root=base_dir, transform=val_test_transforms)
test_base.targets = [label_map[label] for label in test_base.targets]
test_base.samples = [(path, label_map[label]) for path, label in test_base.samples]

val_dataset  = Subset(val_base,  val_idx)
test_dataset = Subset(test_base, test_idx)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,  num_workers=num_workers)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False, num_workers=num_workers)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, num_workers=num_workers)

In [ ]:
class_names = full_dataset.classes
print(f"Classes      : {class_names}")
print(f"Total images : {total}")
print(f"Train        : {train_size} ({train_ratio:.0%})")
print(f"Val          : {val_size}   ({val_ratio:.0%})")
print(f"Test         : {test_size}  ({test_ratio:.0%})")

images, labels = next(iter(train_loader))
print(f"Classes      : ['negative', 'positive']")
print("Label mapping: {0: 'negative (angry, melancholic, sad)', 1: 'positive (happy)'}")

Classes      : ['angry', 'happy', 'melancholic', 'sad']
Total images : 904
Train        : 632 (70%)
Val          : 135   (15%)
Test         : 137  (15%)
Classes      : ['negative', 'positive']
Label mapping: {0: 'negative (angry, melancholic, sad)', 1: 'positive (happy)'}
